#### **Introduction to Importance Sampling and Metropolis-Hastings Monte Carlo** <br> Joshua Pajak, Ph.D. joshua.pajak@umassmed.edu

This notebook is designed to introduce the reader to key concepts underlying **importance sampling** using **Metropolis-Hastings Monte Carlo** simulations of a simple harmonic oscillator. 

Initially, we will sample the ensemble by generating configurations randomly and accepting them with the Metropolis-Hastings criteria. Then, we will implement a importance sampling approach to generating poses. We will see how this approach lets us approximate the ensemble much more quickly and efficiently.

In [ ]:
# --- First we import all the libraries we need ---
import numpy as np  # fast linear algebra
import pandas as pd # dataframes are good
import matplotlib.pyplot as plt # plotting
import seaborn as sns # more plotting
sns.set_theme('notebook')

Let's calculate the theoretical probability distribution functions (PDFs) for a harmonic oscillator in 1D so that we have something to vet our simulations against.

In [ ]:
# --- Make energy function so we can easily calculate energies of states ---
def total_energy_physical(x, v, k, x_center=0.0):
    """
    Calculates the total energy (Hamiltonian) of a ball on a spring.
    Args:
        x (float): position coordinate
        v (float): velocity coordinate
        k (float): spring constant
        x_center (float): the center of the spring (default=0.0)
    Returns:
        float: total energy E
    """
    E_potential = 0.5 * k * (x - x_center)**2
    E_kinetic = 0.5 * v**2
    return E_potential + E_kinetic

In [ ]:
def calculate_theoretical_distribution(k, KT, x_center=0.0, x_lim=10, v_lim=10, resolution=100):
    """
    Generates a DataFrame of theoretical probabilities for the harmonic oscillator
    across a grid of position and velocity values.
    Args:
        k (float): spring constant (reduced units; must be positive)
        KT (float): temperature factor (must be positive)
        x_center (float): center of spring (default=0.0)
        x_lim (float): limit to calculate position (default=10)
        v_lim (float): limit to calculate velocity (default=10)
        resolution (int): how many points in our meshgrid
        
    Returns:
        DataFrame: ['position', 'velocity', 'probability']
    """
    # --- Create a grid of x and v values --
    x_range = np.linspace(x_center - x_lim, x_center + x_lim, resolution)
    v_range = np.linspace(-v_lim, v_lim, resolution)
    X, V = np.meshgrid(x_range, v_range)

    # --- Calculate energy of every point in our meshgrid ---
    Energy = total_energy_physical(X,V,k,x_center)

    # --- Calculate the Boltzmann distribution ---
    prob_density = np.exp(-Energy / KT)

    # --- Normalize by the parition function --- 
    Z = (2 * np.pi * KT) / np.sqrt(k)
    prob_density /= Z

    # 5. Flatten and convert to a tidy DataFrame
    df_theory = pd.DataFrame({
        'position': X.ravel(),
        'velocity': V.ravel(),
        'probability': prob_density.ravel()
    })

    return df_theory

Great! So now will have a sense for what we are after. But now let's imagine we didn't know how to calculate the partition function, so we didn't know how to normalize our distribution. This is where **Metropolis-Hastings Monte Carlo** comes in handy. We will generate new states and **accept** or **reject** those states based on the **energy relative to our current state.** Because we are looking at the _ratio_ of states, the partition function cancels and we can approximate the probability distribution with the correct normalization. 

In [ ]:
# --- Define our acceptance criteria --- 
def acceptance_probability(delta_energy, KT):
    return min(1.0, np.exp(-delta_energy / KT))

First, we will implement a "naive" Metropolis-Hastings approach, where new states are generated truly randomly. To limit our search, we will set bounds $x_{lim}$ and $v_{lim}$, otherwise our search would have to be infinitely broad.  

In [ ]:
def metropolis_naive(k, KT, x_center = 0.0, x_lim = 10, v_lim = 10, num_trials = 10000):
    """
    Calculates the (position, velocity) pairs that correspond to a constant 
    energy E for a ball (mass=1) on a spring (constant k).
    
    This algorithm implements Metropolis acceptance criteria to perform
    a Monte Carlo simulation.

    Args:
        k (float): The spring constant (unitless). Must be positive.
        KT (float): Temperature factor of the system (inverse of beta; reduced units).
        x_center (float): The center of the spring (default 0.0).
        x_lim (float): limit for search in x.
        v_lim (float): limit for search in v.
        num_trials (int): The number of trials to perform.
        
    Returns:
        pd.DataFrame: A history of all proposed states with columns
                      'position', 'velocity', and 'accepted'.
    """

    # --- Initialize our history to keep track ---
    history_data = {
        'position': [],
        'velocity':[],
        'energy':[],
        'accepted':[]
    }

    # --- Initialize our system ---
    x_current, v_current = x_center, 0.0 
    E_current = total_energy_physical(x_current, v_current, k, x_center)

    # --- Iterate over the number of trials num_trials
    for _ in range(num_trials):
        # --- Propose new position and velocity ---
        x_proposed = np.random.uniform(x_center - x_lim, x_center + x_lim)
        v_proposed = np.random.uniform(-v_lim, v_lim)
        
        # --- Calculate the delta energy ---
        E_proposed = total_energy_physical(x_proposed, v_proposed, k, x_center)
        delta_energy = E_proposed - E_current

        # --- Check the relative probability of the proposed move vs. current ---
        P_acc = acceptance_probability(delta_energy, KT)
        
        # --- Throw the die ---
        r = np.random.rand()
        is_accepted = False # we can change this if and when it is accepted
        if r < P_acc:
            x_current, v_current, E_current = x_proposed, v_proposed, E_proposed
            is_accepted = True
        # --- log the simulation ---
        history_data['position'].append(x_proposed)
        history_data['velocity'].append(v_proposed)
        history_data['energy'].append(E_proposed)
        history_data['accepted'].append(is_accepted) 
        
    return pd.DataFrame(history_data)

In [ ]:
# --- Calculate probability distribution from theory and using Metropolis-Hastings ---
k, KT = 1, 1 # define here to ensure same k and KT used in our calculations
df_theory = calculate_theoretical_distribution(k=k, KT=KT)
df_naive = metropolis_naive(k=k, KT=KT, num_trials=100000)


In [ ]:
# --- Set up a 2x2 grid ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Top Left: Bivariate Distribution ---
# plot the simulated data as a histogram
sns.histplot(data=df_naive, 
             x='position', 
             y='velocity', 
             hue='accepted', 
             ax=axes[0, 0])
# plot the theoretical distribution as contour lines
sns.kdeplot(
    data=df_theory,
    x='position',
    y='velocity',
    weights='probability',
    fill=False,
    alpha=0.5,
    ax=axes[0,0],
    cmap='Grays'
)
axes[0, 0].set_title("Position vs Velocity (Proposed States)")

# --- Top Right: Position Histogram ---
sns.histplot(data=df_naive, x='position', hue='accepted', multiple='stack', ax=axes[0, 1])
axes[0, 1].set_title("Position Distribution")

# --- Bottom Left: Energy Histogram ---
sns.histplot(data=df_naive, x='energy', hue='accepted', multiple='stack', ax=axes[1, 0])
axes[1, 0].set_title("Energy Distribution")

# --- Bottom Right: Velocity Histogram ---
sns.histplot(data=df_naive, x='velocity', hue='accepted', multiple='stack', ax=axes[1, 1])
axes[1, 1].set_title("Velocity Distribution")

plt.tight_layout()
plt.show()

This is great! We are sampling a distribution that is truly approximating the theoretical distribution. But at the same time we are wasting a lot of computing power on moves that we reject. This is in essense the same problem we tried to overcome with the Gillespie algorithm. We would like to generate states that are accepted more frequently. In other words, we would like to **sample** the **important** states more often. Can we think of some ways to achieve this?

One way is to make our simulation a Markov Chain. That is, the proposed state will be generated by performing some operation to the current state. Because the current state was accepted, small perturbations away from the current state are also likely to be accepted.

In [ ]:
def metropolis_importance(k, KT, x_center = 0.0, x_lim = 10, v_lim = 10, step_size=2.0, num_trials = 10000):
    """
    Calculates the (position, velocity) pairs that correspond to a constant 
    energy E for a ball (mass=1) on a spring (constant k).
    
    This algorithm implements Metropolis acceptance criteria to perform
    a Monte Carlo simulation.

    Args:
        k (float): The spring constant (unitless). Must be positive.
        KT (float): Temperature factor of the system (inverse of beta; reduced units).
        x_center (float): The center of the spring (default 0.0).
        x_lim (float): limit for search in x.
        v_lim (float): limit for search in v.
        step_size (float): new states will be sampled by nudging current state by
                           a number drawn uniformly from [-step_size, step_size]
        num_trials (int): The number of trials to perform.
        
    Returns:
        pd.DataFrame: A history of all proposed states with columns
                      'position', 'velocity', and 'accepted'.
    """

    # --- Initialize our history to keep track ---
    history_data = {
        'position': [],
        'velocity':[],
        'energy':[],
        'accepted':[]
    }

    # --- Initialize our system ---
    x_current, v_current = x_center, 0.0 
    E_current = total_energy_physical(x_current, v_current, k, x_center)

    # --- Iterate over the number of trials num_trials
    for _ in range(num_trials):
        # --- Propose new position and velocity ---
        x_proposed = x_current + np.random.uniform(-step_size, step_size)
        v_proposed = v_current + np.random.uniform(-step_size, step_size)
        
        # --- Calculate the delta energy ---
        E_proposed = total_energy_physical(x_proposed, v_proposed, k, x_center)
        delta_energy = E_proposed - E_current

        # --- Check the relative probability of the proposed move vs. current ---
        P_acc = acceptance_probability(delta_energy, KT)
        
        # --- Throw the die ---
        r = np.random.rand()
        is_accepted = False # we can change this if and when it is accepted
        if r < P_acc:
            x_current, v_current, E_current = x_proposed, v_proposed, E_proposed
            is_accepted = True
        # --- log the simulation ---
        history_data['position'].append(x_proposed)
        history_data['velocity'].append(v_proposed)
        history_data['energy'].append(E_proposed)
        history_data['accepted'].append(is_accepted) 
        
    return pd.DataFrame(history_data)

In [ ]:
# --- Run simulation with importance sampling ---
df_importance = metropolis_importance(k=k, KT=KT, num_trials=100000)

In [ ]:
# --- Set up a 2x2 grid ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# --- Top Left: Bivariate Distribution ---
# plot the simulated data as a histogram
sns.histplot(data=df_importance, 
             x='position', 
             y='velocity', 
             hue='accepted', 
             ax=axes[0, 0])
# plot the theoretical distribution as contour lines
sns.kdeplot(
    data=df_theory,
    x='position',
    y='velocity',
    weights='probability',
    fill=False,
    alpha=0.5,
    ax=axes[0,0],
    cmap='Grays'
)
axes[0, 0].set_title("Position vs Velocity (Proposed States)")

# --- Top Right: Position Histogram ---
sns.histplot(data=df_importance, x='position', hue='accepted', multiple='stack', ax=axes[0, 1])
axes[0, 1].set_title("Position Distribution")

# --- Bottom Left: Energy Histogram ---
sns.histplot(data=df_importance, x='energy', hue='accepted', multiple='stack', ax=axes[1, 0])
axes[1, 0].set_title("Energy Distribution")

# --- Bottom Right: Velocity Histogram ---
sns.histplot(data=df_importance, x='velocity', hue='accepted', multiple='stack', ax=axes[1, 1])
axes[1, 1].set_title("Velocity Distribution")

plt.tight_layout()
plt.show()

### **Homework for the reader**
1. What happens when step size in importance sampling is too big? Too small?